# INF201 Deliverable 3: RPG Battle Simulator

GitHub repository: https://github.com/fabianfeiler/inf201-h26-deliverable-3

In [1]:
import random
from battle import Battle
from character import Character, Warrior, Mage
from dice import Dice
from items import Weapon

___________________________________
## Class overview

| Class | Module | Purpose | Relationship |
|---|---|---|---|
| `Dice` | `dice.py` | Rolls a random number from 1 to `sides` | Used by `Weapon` |
| `Weapon` | `items.py` | Rolls damage with its `Dice` | Has a `Dice` (composition) |
| `Character` | `character.py` | Base class with HP, damage and attacks | Can have a `Weapon` (composition) |
| `Warrior` | `character.py` | Defensive character with armor | Is a `Character` (inheritance) |
| `Mage` | `character.py` | Uses mana for magic attacks | Is a `Character` (inheritance) |
| `Battle` | `battle.py` | Runs turn-based combat | Has two `Character` objects |

The cell below creates one object of each class and prints its attributes and public methods.

In [2]:
def show(obj: object) -> None:
    """Print the class name, attributes and public methods of an object."""
    attributes = list(vars(obj))
    methods = [name for name in dir(obj) if not name.startswith("_") and name not in attributes]
    print(type(obj).__name__)
    print(f"  Attributes: {attributes}")
    print(f"  Methods and properties: {methods}\n")


sword = Weapon("Sword", Dice(sides=12))
objects = [Dice(), sword, Character("Hero"), Warrior("Warrior", weapon=sword), Mage("Mage"), Battle(Character("A"), Character("B"))]

for obj in objects:
    show(obj)

Dice
  Attributes: ['sides']
  Methods and properties: ['roll']

Weapon
  Attributes: ['name', 'damage_dice']
  Methods and properties: ['roll_damage']

Character
  Attributes: ['name', '_hp', 'hp_max', 'damage', 'weapon']
  Methods and properties: ['attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']

Warrior
  Attributes: ['name', '_hp', 'hp_max', 'damage', 'weapon', 'armor']
  Methods and properties: ['attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']

Mage
  Attributes: ['name', '_hp', 'hp_max', 'damage', 'weapon', 'mana', 'mana_max']
  Methods and properties: ['attack', 'heal', 'hp', 'introduce', 'is_alive', 'take_damage']

Battle
  Attributes: ['hero', 'enemy', 'max_rounds', 'round']
  Methods and properties: ['play_round', 'simulate_battle']



**Result:** `Warrior` and `Mage` have all the methods from `Character` through inheritance, and only add their own attributes (`armor`, `mana`, `mana_max`). `_hp` is listed as an attribute, but is only read through the `hp` property and changed by `take_damage()` and `heal()`. Special methods like `__call__` and `__repr__` in `Dice` are not listed, since names starting with `_` are filtered out. The same goes for `_attack()` in `Battle`, which is a private helper method.

-------
## Task 1: Update the Character class

`attack()` used to change the target's `_hp` directly. Now it calls `target.take_damage()`, so each character only changes its own HP. We also added `heal()` and the property `is_alive`.

In [3]:
enemy = Character("Enemy", hp_max=25)
hero = Character("Hero")

hero.attack(enemy)
hero.attack(enemy)
hero.attack(enemy)
print(enemy.hp, enemy.is_alive)

enemy.heal(100)
print(enemy.hp)

0 False
25


**Result:** After three attacks the enemy has 0 HP, not -5, and `is_alive` is `False`. Healing 100 only brings HP back to 25, which is `hp_max`. HP always stays between 0 and `hp_max`.

___________________________________
## Task 2: Dice

`Dice` can create any n-sided dice (n >= 2, default 6). `roll()` returns a random number from 1 to `sides` using `random.randint`. `__call__` makes the object callable, so `d20()` does the same as `d20.roll()`. `__repr__` shows the dice as `Dice(sides=20)`.

In [4]:
random.seed(42)  # Fixed seed so the results are reproducible

d6 = Dice()
d20 = Dice(sides=20)
print(d6, d20)
print([d6.roll() for _ in range(10)])
print([d20() for _ in range(10)])

try:
    Dice(sides=1)
except ValueError as err:
    print(f"ValueError: {err}")

Dice(sides=6) Dice(sides=20)
[6, 1, 1, 6, 3, 2, 2, 2, 6, 1]
[18, 3, 19, 14, 2, 1, 3, 7, 8, 17]
ValueError: A dice must have at least 2 sides


**Result:** The default dice has 6 sides, and all rolls stay between 1 and `sides`. Calling `d20()` works the same as `d20.roll()`. Creating a dice with 1 side raises a `ValueError`, so the rule n >= 2 is enforced.

--------------------
## Task 3: Weapon

A `Weapon` has a name and a `Dice`. `roll_damage()` rolls the dice. If a character has a weapon, `attack()` uses the weapon damage instead of the base damage.

In [5]:
sword = Weapon("Sword", Dice(sides=12))
hero = Character("Hero", weapon=sword)
enemy = Character("Enemy")

hero.attack(enemy)
print(enemy.hp)

90


**Result:** The enemy loses between 1 and 12 HP, depending on the dice roll, instead of the fixed 10 damage.

------
## Task 4: Warrior and Mage

`Warrior` and `Mage` are subclasses of `Character`. A `Warrior` has armor that reduces incoming damage. A `Mage` uses 10 mana for a magic attack with double damage. Without enough mana it uses a normal attack and restores 5 mana.

In [6]:
warrior = Warrior("Warrior")
mage = Mage("Mage")

for i in range(5):
    mage.attack(warrior)
    print(warrior.hp, mage.mana)

85 20
70 10
55 0
50 5
45 10


**Result:** The first three attacks are magic: 20 damage minus 5 armor gives 15. Then the mana is too low, so the mage uses normal attacks: 10 minus 5 armor gives 5 damage, and the mage gets 5 mana back each time.

-------------
## Task 5: Battle

`Battle` manages turn-based combat between a hero and an enemy. In `play_round()` the hero attacks first, and the enemy strikes back if it is still alive. `simulate_battle()` runs rounds in a while loop until one character dies, and returns the winner.

The damage shown is measured as HP before minus HP after the attack, so it includes the Warrior's armor reduction. We also added a `max_rounds` limit (default 100), since two characters with high armor and low damage could otherwise fight forever.

In [7]:
random.seed(42)

hero = Warrior("Aragorn", weapon=Weapon("Sword", Dice(sides=12)))
enemy = Mage("Saruman")

battle = Battle(hero, enemy)
winner = battle.simulate_battle()


Round 1
Aragorn hits Saruman for 11 damage (89 HP left)
Saruman hits Aragorn for 15 damage (85 HP left)

Round 2
Aragorn hits Saruman for 2 damage (87 HP left)
Saruman hits Aragorn for 15 damage (70 HP left)

Round 3
Aragorn hits Saruman for 1 damage (86 HP left)
Saruman hits Aragorn for 15 damage (55 HP left)

Round 4
Aragorn hits Saruman for 12 damage (74 HP left)
Saruman hits Aragorn for 5 damage (50 HP left)

Round 5
Aragorn hits Saruman for 5 damage (69 HP left)
Saruman hits Aragorn for 5 damage (45 HP left)

Round 6
Aragorn hits Saruman for 4 damage (65 HP left)
Saruman hits Aragorn for 15 damage (30 HP left)

Round 7
Aragorn hits Saruman for 4 damage (61 HP left)
Saruman hits Aragorn for 5 damage (25 HP left)

Round 8
Aragorn hits Saruman for 3 damage (58 HP left)
Saruman hits Aragorn for 5 damage (20 HP left)

Round 9
Aragorn hits Saruman for 12 damage (46 HP left)
Saruman hits Aragorn for 15 damage (5 HP left)

Round 10
Aragorn hits Saruman for 2 damage (44 HP left)
Saruman h

**Result:** Saruman wins after 10 rounds with 44 HP left. The Mage's damage follows a fixed pattern: magic attacks deal 20 minus 5 armor = 15, and base attacks deal 10 minus 5 = 5. Since the Mage has no weapon, this pattern is the same every time, and the Warrior always dies in round 10. Aragorn would need an average of 10 damage per hit to win, but a d12 sword only averages 6.5.

In [8]:
tank_a = Warrior("Tank A", damage=5, armor=5)
tank_b = Warrior("Tank B", damage=5, armor=5)

draw = Battle(tank_a, tank_b, max_rounds=10).simulate_battle()


Round 1
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 2
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 3
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 4
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 5
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 6
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 7
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 8
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 9
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (100 HP left)

Round 10
Tank A hits Tank B for 0 damage (100 HP left)
Tank B hits Tank A for 0 damage (10

**Result:** Both Warriors deal 5 damage and have 5 armor, so every attack deals 0 damage. Without `max_rounds` the while loop would never end. With the limit, the battle stops after 10 rounds and `simulate_battle()` returns `None` to indicate a draw.